# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# dataset.metadata is an object, access its attributes with .
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs by exploring the schema.

In [ ]:
# List all record sets, their @ids, and their field @ids
print('Available Record Sets:')
record_sets = dataset.metadata.record_sets
record_set_ids = []
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}, @id: {rs.id}")
    record_set_ids.append(rs.id)
    field_ids = [f.id for f in rs.fields]
    print(f"    Fields: {[f'@id: {f.id}, name: {f.name}' for f in rs.fields]}")

print("\nFirst 2 records from each RecordSet:")
for rs in record_sets:
    print(f'-- {rs.name} (@id: {rs.id})')
    try:
        for i, rec in enumerate(dataset.records(record_set=rs.id)):
            print(rec)
            if i >= 1:
                break
    except Exception as e:
        print(f'Unable to load records for RecordSet @id {rs.id}: {e}')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview. All field and record set references are via their `@id`.

In [ ]:
# Prepare to extract all available RecordSets by id as DataFrames
dataframes = {}
# We'll use `record_set_ids` as collected in previous cell
for rs_id in record_set_ids:
    # Load records for that record set
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"Loaded DataFrame for RecordSet @id: {rs_id}, shape: {df.shape}")
        else:
            print(f"No records found for RecordSet @id: {rs_id}")
    except Exception as e:
        print(f"Failed to load DataFrame for RecordSet @id: {rs_id}: {e}")

# For demonstration, let's choose the first record set for EDA
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f'Columns in main DataFrame (@id: {main_record_set_id}):')
    print(dataframes[main_record_set_id].columns.tolist())
    print(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes. All field references are by `@id` (as the DataFrame columns are named by them).

In [ ]:
# Example: pick numeric fields (e.g., coverage or MW)
# List candidate numeric fields by @id and example values
df = dataframes[main_record_set_id]
numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
if not numeric_columns:
    # Try infer numbers from string columns
    for col in df.columns:
        try:
            df[col+'_float'] = pd.to_numeric(df[col], errors='coerce')
            if df[col+'_float'].notnull().sum() > 0:
                numeric_columns.append(col+'_float')
        except Exception as e:
            continue
print(f"Available numeric fields (for EDA, by column name / @id): {numeric_columns}")

# Let's select the first numeric field
if numeric_columns:
    numeric_field = numeric_columns[0]
    # Filtering for values above a threshold (e.g. mean)
    threshold = df[numeric_field].mean()
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize this field
    filtered_df[f'{numeric_field}_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f'{numeric_field}_normalized']].head())

    # Attempt to group by a non-numeric field (e.g. 'Protein name' or accession)
    group_candidates = [c for c in df.columns if c not in numeric_columns]
    if group_candidates:
        group_field = group_candidates[0]
        try:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped mean {numeric_field} by {group_field}:")
            print(grouped_df.head())
        except Exception as e:
            print(f"Could not group by {group_field}: {e}")
else:
    print("No numeric fields available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualization with matplotlib (and optionally seaborn)
import matplotlib.pyplot as plt

if 'numeric_field' in locals() and numeric_field in filtered_df.columns:
    # Distribution histogram
    plt.figure(figsize=(7,4))
    filtered_df[numeric_field].hist(bins=20)
    plt.title(f"Distribution of {numeric_field} for filtered records")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # Normalized distribution
    if f'{numeric_field}_normalized' in filtered_df.columns:
        plt.figure(figsize=(7,4))
        filtered_df[f'{numeric_field}_normalized'].hist(bins=20)
        plt.title(f"Normalized Distribution of {numeric_field}")
        plt.xlabel(f'{numeric_field}_normalized')
        plt.ylabel("Frequency")
        plt.show()

    # Simple scatter plot if other numeric field exists
    if len(numeric_columns) > 1:
        other_numeric = numeric_columns[1]
        if other_numeric in filtered_df.columns:
            plt.figure(figsize=(6, 6))
            plt.scatter(filtered_df[numeric_field], filtered_df[other_numeric])
            plt.xlabel(numeric_field)
            plt.ylabel(other_numeric)
            plt.title(f"Scatter plot: {numeric_field} vs {other_numeric}")
            plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

**Key Observations:**
- This dataset contains mass spectrometry-based protein data from human extracellular vesicles. All tables and fields were referenced by their Croissant schema `@id`s for reproducibility.
- Numeric fields (such as protein coverage or molecular weight) can be filtered and normalized for statistical analysis, and the dataset supports exploratory groupings, e.g., by protein accession or sample type.
- Visualization of numeric distributions (e.g., histograms, scatter plots) provides further biological and experimental insights.

Further analysis may include:
- Deeper feature selection from provided fields
- Integration with identifier databases for richer annotation
- Advanced clustering or multivariate analysis depending on research needs.
